# Simple LLM Pipeline (Minimal GPT from Scratch)
This notebook shows the **simplest end‑to‑end pipeline** for a tiny LLM:
1. Load text
2. Tokenize
3. Build embeddings
4. Implement attention
5. Build a tiny GPT model
6. Train on next‑token prediction
7. Generate text

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


## 1. Load Simple Training Text

In [2]:
# text = """
# Machine learning is fun.
# Deep learning builds powerful models.
# Language models learn patterns in text.
# """


In [3]:
# Use real text for better Vocabulary
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

## 2. Tokenization (character level for simplicity)

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(vocab_size)


65


## 3. Create Training Batches

In [5]:
# block_size = 8
# block_size = 16
block_size = 64
# Increase Context Window

def get_batch():
    # ix = torch.randint(len(data)-block_size, (4,)) -> batch size = 4, very tiny when transfomer 
    # bigger
    batch_size = 32

    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y


## 4. Self‑Attention

In [6]:
class SelfAttention(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.key = nn.Linear(n_embd, n_embd)
        self.query = nn.Linear(n_embd, n_embd)
        self.value = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        K = self.key(x)
        Q = self.query(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2,-1) / math.sqrt(x.size(-1))
        weights = F.softmax(scores, dim=-1)

        return weights @ V


## 5. Transformer Block

In [7]:
class Block(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.attn = SelfAttention(n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd)
        )

    def forward(self, x):
        x = x + self.attn(x)
        x = x + self.ff(x)
        return x


## 6. Tiny GPT Model

In [8]:
class TinyGPT(nn.Module):
    # def __init__(self, vocab_size, n_embd=32):
    # def __init__(self, vocab_size, n_embd=64):
    def __init__(self, vocab_size, n_embd=128):
        # Slightly Bigger Model
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)

        # self.block = Block(n_embd) # ← only ONE block
        # So currently you have:
        # layers = 1
        # attention heads = 1

        # That’s why you don’t see them.
        # You replace it with multiple blocks.
        self.blocks = nn.Sequential(
            Block(n_embd),
            Block(n_embd),
            Block(n_embd),
            Block(n_embd)
        )

        # Now you have:
        # layers = 4
        
        self.ln = nn.LayerNorm(n_embd)

        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B,T = idx.shape

        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T))

        x = tok + pos

        # x = self.block(x)
        x = self.blocks(x)
        
        x = self.ln(x)

        logits = self.head(x)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


## 7. Train Model

In [9]:
model = TinyGPT(vocab_size)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) 
# High LR can cause weird training behavior.
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# from range 500 to 5000, mean train longer
# for step in range(5000):
for step in range(20000): # TRAINING STEP

    xb, yb = get_batch()

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(step, loss.item())


0 4.345432281494141
100 2.6146399974823
200 0.43794354796409607
300 0.08638939261436462
400 0.06228535994887352
500 0.0631881058216095
600 0.045753899961709976
700 0.044271714985370636
800 0.04562632739543915
900 0.04386388510465622
1000 0.033335670828819275
1100 0.048421215265989304
1200 0.043562520295381546
1300 0.03510231152176857
1400 0.037468839436769485
1500 0.0342145599424839
1600 0.037436217069625854
1700 0.0337999053299427
1800 0.031828995794057846
1900 0.036359529942274094
2000 0.03577225282788277
2100 0.0448664054274559
2200 0.03314890339970589
2300 0.04410950094461441
2400 0.040895797312259674
2500 0.03615633398294449
2600 0.024290530011057854
2700 0.03708168864250183
2800 0.030879849568009377
2900 0.0299222394824028
3000 0.029280712828040123
3100 0.04045147821307182
3200 0.026299281045794487
3300 0.03402074798941612
3400 0.03639335557818413
3500 0.0353255420923233
3600 0.03642130270600319
3700 0.02882886864244938
3800 0.03552240505814552
3900 0.03185227885842323
4000 0.032

## 8. Generate Text

In [10]:
def generate(model, start, length=100):

    idx = torch.tensor([encode(start)], dtype=torch.long)

    for _ in range(length):

        idx_cond = idx[:,-block_size:]

        logits,_ = model(idx_cond)

        logits = logits[:,-1,:]

        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs,1)

        idx = torch.cat((idx,next_id),dim=1)

    return decode(idx[0].tolist())


print(generate(model, "Machine"))


MachineinehIWmXhemhimoWWemhXRQRWARmRRRRmmWMWWARKRAmRRWmRRRWmRWXWstlant,
Is the to cont Romeo! your hath, I 


In [11]:
generate(model, "KING", length=300)

"KINGKINKKIVKIIVlIIFKKKIIIIIIIIIIIIIIIIIIKIIIIIIIIIIIIIIIIIIIIIII:\nA seem, IIIIIIIIII mIIIIIIIIIIIIII:\nIIIIA dom, will thy noss? thou murk'd.\n\nGLOONIO:\nI' procent compor, lady; leavef liobaly,\nVenters:\nI hath you, Mercience\nfroW the my ester soo; and of Nurstant:\nI am the moin makes, sir, for\nCond bring "

One very interesting thing

You are basically reproducing what happens when people build their first transformer.

The moment where output changes from:

gibberish

to

almost words

is the first proof the attention mechanism is working.

Notice what the model already learned:

Capital letters starting lines

Apostrophes (I'ran)

Sentence punctuation

Word-like shapes

Line breaks like dialogue

Short answer: don’t jump to GPT-2 yet.
First scale your tiny model step-by-step. That’s exactly how most tutorials (like nanoGPT) teach it.

Right now your model is learning correctly, but it's still a toy GPT.

Where You Are Now

Your current config:

block_size = 16
n_embd = 64
training_steps = 5000
layers = 1

This is roughly:

~100K parameters

Very tiny.

What a GPT-2–like Model Looks Like

Example: GPT-2 small

layers = 12
heads = 12
embedding = 768
context window = 1024
parameters ≈ 124M

That is ~1000× bigger than your model.

You cannot realistically train that in a Kaggle notebook from scratch.

The Correct Learning Path (What I Recommend)

Instead of jumping to GPT-2, scale like this:

Stage 1 — Your current tiny GPT
block_size = 16
n_embd = 64
layers = 1

Goal:
✔ pipeline works
✔ text generation works

You already reached this stage.

Stage 2 — Small GPT

Upgrade to:

block_size = 64
n_embd = 128
layers = 4

Approx:

~3–5 million parameters

Now generation becomes much more coherent.

Example output:

KING:
My lord, what news from the court?

QUEEN:
The prince has fled the city tonight.